# EDA-03 · Inventory Health
Stockout frequency, overstock analysis, fill_rate vs stockout_days, days_of_supply heatmap.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 10,
})

inv   = pd.read_csv('inventory.csv', parse_dates=['snapshot_date'])
prods = pd.read_csv('products.csv')

inv['year']  = inv['snapshot_date'].dt.year
inv['month'] = inv['snapshot_date'].dt.month
# inventory.csv đã có product_name, category, segment, year, month sẵn

CAT_ORDER = ['Streetwear','Outdoor','Casual','GenZ']
CAT_COLORS = ['#4C72B0','#DD8452','#55A868','#C44E52']

print(f'Inventory snapshots : {len(inv):,}')
print(f'Date range          : {inv["snapshot_date"].min().date()} -> {inv["snapshot_date"].max().date()}')
print(f'Unique products     : {inv["product_id"].nunique():,}')
print(f'Unique months       : {inv["snapshot_date"].nunique()}')
inv.head(3)


## 1. Overall Flag Distribution

In [ ]:
total = len(inv)
so_rate  = inv['stockout_flag'].mean()  * 100
ov_rate  = inv['overstock_flag'].mean() * 100
both     = ((inv['stockout_flag']==1) & (inv['overstock_flag']==1)).mean() * 100
neither  = ((inv['stockout_flag']==0) & (inv['overstock_flag']==0)).mean() * 100

print(f'Total snapshots      : {total:,}')
print(f'Stockout flag = 1    : {inv["stockout_flag"].sum():,}  ({so_rate:.1f}%)')
print(f'Overstock flag = 1   : {inv["overstock_flag"].sum():,}  ({ov_rate:.1f}%)')
print(f'Both flags = 1       : {int(both*total/100):,}  ({both:.1f}%)  <- stock-out AND overstock simultaneously')
print(f'Neither flag         : {int(neither*total/100):,}  ({neither:.1f}%)')
print(f'Reorder flag = 1     : {inv["reorder_flag"].sum():,}  ({inv["reorder_flag"].mean()*100:.1f}%)')

labels = ['Stockout only','Overstock only','Both','Neither']
vals = [
    ((inv['stockout_flag']==1) & (inv['overstock_flag']==0)).sum(),
    ((inv['stockout_flag']==0) & (inv['overstock_flag']==1)).sum(),
    ((inv['stockout_flag']==1) & (inv['overstock_flag']==1)).sum(),
    ((inv['stockout_flag']==0) & (inv['overstock_flag']==0)).sum(),
]
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(labels, vals, color=['#C44E52','#DD8452','#9467BD','#55A868'])
ax.set_ylabel('Snapshots'); ax.set_title('Inventory flag distribution')
for bar, v in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+200,
            f'{v:,}\n({v/total*100:.1f}%)', ha='center', fontsize=9)
plt.tight_layout(); plt.show()

## 2. Stockout Frequency

In [ ]:
# -- by category --
cat_so = inv.groupby('category').agg(
    snapshots      = ('stockout_flag','count'),
    stockout_snaps = ('stockout_flag','sum'),
    avg_stockout_days = ('stockout_days','mean'),
    pct_months_stockout = ('stockout_flag','mean'),
).reset_index()
cat_so['pct_months_stockout'] *= 100
print('Stockout by category:')
print(cat_so.round(2).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(cat_so['category'], cat_so['pct_months_stockout'],
            color=CAT_COLORS[:len(cat_so)])
axes[0].set_ylabel('% months in stockout'); axes[0].set_title('Stockout rate by category')
for i, v in enumerate(cat_so['pct_months_stockout']):
    axes[0].text(i, v+0.3, f'{v:.1f}%', ha='center', fontsize=9)

axes[1].bar(cat_so['category'], cat_so['avg_stockout_days'],
            color=CAT_COLORS[:len(cat_so)])
axes[1].set_ylabel('Avg stockout days/month'); axes[1].set_title('Avg stockout days by category')
for i, v in enumerate(cat_so['avg_stockout_days']):
    axes[1].text(i, v+0.02, f'{v:.2f}', ha='center', fontsize=9)
plt.tight_layout(); plt.show()

## 3. Stockout Trend by Year & Month

In [ ]:
# Yearly trend
yr_so = inv.groupby('year').agg(
    stockout_rate = ('stockout_flag','mean'),
    avg_days      = ('stockout_days','mean'),
).reset_index()
yr_so['stockout_rate'] *= 100

# Monthly trend (avg across years)
mo_so = inv.groupby('month').agg(
    stockout_rate = ('stockout_flag','mean'),
    avg_days      = ('stockout_days','mean'),
).reset_index()
mo_so['stockout_rate'] *= 100
MONTH_LABELS = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(yr_so['year'], yr_so['stockout_rate'], marker='o', color='#C44E52')
axes[0].fill_between(yr_so['year'], yr_so['stockout_rate'], alpha=0.15, color='#C44E52')
axes[0].set_xlabel('Year'); axes[0].set_ylabel('Stockout rate %')
axes[0].set_title('Stockout rate trend by year')
for _, row in yr_so.iterrows():
    axes[0].text(row['year'], row['stockout_rate']+0.3, f'{row["stockout_rate"]:.1f}%', ha='center', fontsize=8)

axes[1].bar(range(1,13), mo_so['stockout_rate'], color='#C44E52', alpha=0.8)
axes[1].set_xticks(range(1,13)); axes[1].set_xticklabels(MONTH_LABELS)
axes[1].set_ylabel('Stockout rate %'); axes[1].set_title('Stockout rate by calendar month (avg all years)')
for i, v in enumerate(mo_so['stockout_rate']):
    axes[1].text(i+1, v+0.3, f'{v:.1f}%', ha='center', fontsize=7)
plt.tight_layout(); plt.show()

## 4. Top Products by Stockout Frequency

In [ ]:
prod_so = inv.groupby(['product_id','product_name','category','segment']).agg(
    n_months          = ('stockout_flag','count'),
    months_stockout   = ('stockout_flag','sum'),
    total_stockout_days = ('stockout_days','sum'),
    avg_fill_rate     = ('fill_rate','mean'),
).reset_index()
prod_so['stockout_rate_%'] = prod_so['months_stockout'] / prod_so['n_months'] * 100

top20_so = prod_so.sort_values('months_stockout', ascending=False).head(20)
print('Top 20 products by stockout months:')
print(top20_so[['product_name','category','segment','months_stockout','stockout_rate_%',
                'total_stockout_days','avg_fill_rate']].to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(top20_so['product_name'].str[:25], top20_so['months_stockout'],
        color=[CAT_COLORS[CAT_ORDER.index(c)] if c in CAT_ORDER else 'grey'
               for c in top20_so['category']])
ax.set_xlabel('Months in stockout'); ax.set_title('Top 20 products: stockout months')
ax.invert_yaxis()
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=CAT_COLORS[i], label=CAT_ORDER[i]) for i in range(len(CAT_ORDER))]
ax.legend(handles=legend_elements, loc='lower right', fontsize=8)
plt.tight_layout(); plt.show()

## 5. Overstock Analysis

In [ ]:
# Overstock by category
cat_ov = inv.groupby('category').agg(
    overstock_rate   = ('overstock_flag','mean'),
    avg_stock_on_hand = ('stock_on_hand','mean'),
    avg_units_sold   = ('units_sold','mean'),
    avg_sell_through = ('sell_through_rate','mean'),
).reset_index()
cat_ov['overstock_rate'] *= 100
print('Overstock by category:')
print(cat_ov.round(3).to_string(index=False))

# Top overstock products
prod_ov = inv.groupby(['product_id','product_name','category','segment']).agg(
    n_months           = ('overstock_flag','count'),
    months_overstock   = ('overstock_flag','sum'),
    avg_stock_on_hand  = ('stock_on_hand','mean'),
    avg_sell_through   = ('sell_through_rate','mean'),
    avg_days_of_supply = ('days_of_supply','mean'),
).reset_index()
prod_ov['overstock_rate_%'] = prod_ov['months_overstock'] / prod_ov['n_months'] * 100

top20_ov = prod_ov.sort_values('months_overstock', ascending=False).head(20)
print('\nTop 20 products by overstock months:')
print(top20_ov[['product_name','category','segment','months_overstock',
                'overstock_rate_%','avg_stock_on_hand','avg_sell_through']].to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(cat_ov['category'], cat_ov['overstock_rate'],
            color=CAT_COLORS[:len(cat_ov)])
axes[0].set_ylabel('% months overstock'); axes[0].set_title('Overstock rate by category')
for i, v in enumerate(cat_ov['overstock_rate']):
    axes[0].text(i, v+0.3, f'{v:.1f}%', ha='center', fontsize=9)

axes[1].bar(cat_ov['category'], cat_ov['avg_sell_through'],
            color=CAT_COLORS[:len(cat_ov)])
axes[1].set_ylabel('Avg sell-through rate'); axes[1].set_title('Avg sell-through rate by category')
for i, v in enumerate(cat_ov['avg_sell_through']):
    axes[1].text(i, v+0.002, f'{v:.3f}', ha='center', fontsize=9)
plt.tight_layout(); plt.show()

## 6. Fill Rate vs Stockout Days

In [ ]:
print('fill_rate summary:')
print(inv['fill_rate'].describe().round(4).to_string())
print('\nstockout_days summary:')
print(inv['stockout_days'].describe().round(2).to_string())

r = inv['fill_rate'].corr(inv['stockout_days'])
print(f'\nCorrelation fill_rate vs stockout_days: r = {r:.4f}')

# Scatter (sample 5000 rows để chart nhanh)
sample = inv.sample(min(5000, len(inv)), random_state=42)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# scatter
for cat, grp in sample.groupby('category'):
    idx = CAT_ORDER.index(cat) if cat in CAT_ORDER else -1
    col = CAT_COLORS[idx] if idx >= 0 else 'grey'
    axes[0].scatter(grp['stockout_days'], grp['fill_rate'],
                    alpha=0.3, s=8, color=col, label=cat)
axes[0].set_xlabel('Stockout days'); axes[0].set_ylabel('Fill rate')
axes[0].set_title(f'Fill rate vs Stockout days (r={r:.3f})')
axes[0].legend(fontsize=7, markerscale=2)

# fill_rate distribution
axes[1].hist(inv['fill_rate'], bins=50, color='#4C72B0', edgecolor='white', linewidth=0.3)
axes[1].set_xlabel('Fill rate'); axes[1].set_ylabel('Count')
axes[1].set_title('Fill rate distribution')

# stockout_days distribution (excl 0)
so_nonzero = inv[inv['stockout_days'] > 0]['stockout_days']
axes[2].hist(so_nonzero, bins=range(0, int(so_nonzero.max())+2),
             color='#C44E52', edgecolor='white', linewidth=0.3)
axes[2].set_xlabel('Stockout days'); axes[2].set_ylabel('Count')
axes[2].set_title('Stockout days distribution (excl 0)')
plt.tight_layout(); plt.show()

## 7. Fill Rate by Category & Year

In [ ]:
fr_cat_yr = inv.groupby(['year','category'])['fill_rate'].mean().unstack()
print('Avg fill_rate by category × year:')
print(fr_cat_yr.round(4).to_string())

fig, ax = plt.subplots(figsize=(11, 4))
for i, cat in enumerate(fr_cat_yr.columns):
    ax.plot(fr_cat_yr.index, fr_cat_yr[cat], marker='o',
            label=cat, color=CAT_COLORS[i % len(CAT_COLORS)])
ax.set_xlabel('Year'); ax.set_ylabel('Avg fill rate')
ax.set_title('Fill rate trend by category')
ax.legend(title='Category'); ax.set_ylim(0.9, 1.01)
plt.tight_layout(); plt.show()

## 8. Days of Supply — Heatmap (Category × Month)

In [ ]:
dos_heat = inv.groupby(['category','month'])['days_of_supply'].mean().unstack()
print('Avg days_of_supply (category × calendar month):')
print(dos_heat.round(1).to_string())

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(dos_heat.values, cmap='RdYlGn', aspect='auto')
ax.set_xticks(range(12)); ax.set_xticklabels(MONTH_LABELS)
ax.set_yticks(range(len(dos_heat.index))); ax.set_yticklabels(dos_heat.index)
ax.set_title('Avg days of supply: category × month')
plt.colorbar(im, ax=ax, label='Days of supply')
for i in range(dos_heat.shape[0]):
    for j in range(dos_heat.shape[1]):
        v = dos_heat.values[i, j]
        ax.text(j, i, f'{v:.0f}', ha='center', va='center', fontsize=9,
                color='white' if v < dos_heat.values.min() + (dos_heat.values.max()-dos_heat.values.min())*0.35 else 'black')
plt.tight_layout(); plt.show()

## 9. Days of Supply — Heatmap (Year × Month)

In [ ]:
dos_yr_mo = inv.groupby(['year','month'])['days_of_supply'].mean().unstack()
print('Avg days_of_supply (year × month):')
print(dos_yr_mo.round(1).to_string())

fig, ax = plt.subplots(figsize=(13, 5))
im = ax.imshow(dos_yr_mo.values, cmap='RdYlGn', aspect='auto')
ax.set_xticks(range(12)); ax.set_xticklabels(MONTH_LABELS)
ax.set_yticks(range(len(dos_yr_mo.index))); ax.set_yticklabels(dos_yr_mo.index)
ax.set_title('Avg days of supply: year × month')
plt.colorbar(im, ax=ax, label='Days of supply')
for i in range(dos_yr_mo.shape[0]):
    for j in range(dos_yr_mo.shape[1]):
        v = dos_yr_mo.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f'{v:.0f}', ha='center', va='center', fontsize=8,
                    color='white' if v < dos_yr_mo.values[~np.isnan(dos_yr_mo.values)].min()
                                   + (dos_yr_mo.values[~np.isnan(dos_yr_mo.values)].max()
                                      - dos_yr_mo.values[~np.isnan(dos_yr_mo.values)].min())*0.35
                    else 'black')
plt.tight_layout(); plt.show()

## 10. Simultaneous Stockout & Overstock — Paradox Check

In [ ]:
paradox = inv[(inv['stockout_flag']==1) & (inv['overstock_flag']==1)].copy()
print(f'Snapshots with BOTH stockout & overstock flags: {len(paradox):,} ({len(paradox)/len(inv)*100:.1f}%)')
print()
print('Stats for paradox snapshots:')
print(paradox[['stock_on_hand','units_sold','stockout_days','days_of_supply',
               'fill_rate','sell_through_rate']].describe().round(3).to_string())

print('\nParadox by category:')
print(paradox.groupby('category')[['stock_on_hand','stockout_days','days_of_supply','fill_rate']].mean().round(3).to_string())

## Summary

In [ ]:
print('=== Inventory Health Summary ===')
print(f'Stockout flag rate   : {inv["stockout_flag"].mean()*100:.1f}%  ({inv["stockout_flag"].sum():,} / {len(inv):,} snapshots)')
print(f'Overstock flag rate  : {inv["overstock_flag"].mean()*100:.1f}%  ({inv["overstock_flag"].sum():,} / {len(inv):,} snapshots)')
print(f'Both flags           : {((inv["stockout_flag"]==1)&(inv["overstock_flag"]==1)).sum():,}  ({((inv["stockout_flag"]==1)&(inv["overstock_flag"]==1)).mean()*100:.1f}%)')
print(f'Avg fill_rate        : {inv["fill_rate"].mean():.4f}')
print(f'Avg stockout_days    : {inv["stockout_days"].mean():.2f} days/month')
print(f'Avg days_of_supply   : {inv["days_of_supply"].mean():.1f} days')
print()

worst_so = prod_so.sort_values('months_stockout', ascending=False).iloc[0]
worst_ov = prod_ov.sort_values('months_overstock', ascending=False).iloc[0]
print(f'Most stockout product: {worst_so["product_name"]}  ({worst_so["months_stockout"]} months, {worst_so["stockout_rate_%"]:.1f}%)')
print(f'Most overstock product: {worst_ov["product_name"]}  ({worst_ov["months_overstock"]} months, {worst_ov["overstock_rate_%"]:.1f}%)')
print()
print('Fill rate by category:')
print(inv.groupby('category')['fill_rate'].mean().round(4).to_string())
print()
print('Days of supply by category:')
print(inv.groupby('category')['days_of_supply'].agg(['mean','min','max']).round(1).to_string())